# Baseline: YOLO11 Nano + Green AI
Najpierw wykonaj `prep_dataset.ipynb`. Uruchom notebook z katalogu `garbage-yolo` (Python 3.10+).
Model `yolo11n.pt` uczy się detekcji 6 klas z `data.yaml`; przy pierwszym uruchomieniu pobierze wagi.
`model.train()` zawiera pętlę epok/batchy, obliczanie loss, backpropagation, walidację oraz zapis wag.
CodeCarbon zapisuje szacowane CO₂eq, energię i czas całego wywołania treningowego (łącznie z walidacją wewnętrzną); późniejszy test jest poza pomiarem.
`COUNTRY = "POL"` ustaw zgodnie z fizyczną lokalizacją komputera/serwera. Pomiar obejmuje całą maszynę, więc inne procesy wpływają na wynik.
Na Apple Silicon pomiar energii GPU może być niepełny lub wymagać uprawnień systemowych; CO₂ to estymacja, nie pomiar z licznika.
Dokumentacja: [YOLO11](https://docs.ultralytics.com/models/yolo11/), [trening](https://docs.ultralytics.com/modes/train/), [CodeCarbon](https://docs.codecarbon.io/latest/reference/output/).

In [ ]:
%pip install -q "ultralytics>=8.3,<9" "codecarbon>=3,<4" "pandas>=2,<4"

In [ ]:
from pathlib import Path
from datetime import datetime
from importlib.metadata import version
import json, uuid
import pandas as pd
import torch
from ultralytics import YOLO
from codecarbon import OfflineEmissionsTracker

ROOT = Path.cwd()
DATA = ROOT / "dataset_clean/data.yaml"
assert DATA.is_file(), "Najpierw wykonaj prep_dataset.ipynb."
DEVICE = 0 if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
EPOCHS, BATCH, IMGSZ, SEED = 50, 8, 640, 42
COUNTRY = "POL"  # Zmień, jeżeli trenujesz np. na serwerze poza Polską.
RUN_NAME = "baseline_" + datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6]
RUN = ROOT / "runs" / RUN_NAME
RUN.mkdir(parents=True)
(RUN / "environment.json").write_text(json.dumps({
    "packages": {p: version(p) for p in ("ultralytics", "torch", "codecarbon")},
    "device": str(DEVICE), "country": COUNTRY, "seed": SEED,
}, indent=2))
print("Urządzenie:", DEVICE, "| Wyniki:", RUN)

In [ ]:
model = YOLO("yolo11n.pt")
tracker = OfflineEmissionsTracker(
    project_name=RUN_NAME, country_iso_code=COUNTRY,
    output_dir=str(RUN), output_file="emissions.csv",
    measure_power_secs=15, tracking_mode="machine",
    save_to_file=True, save_to_api=False, log_level="warning",
)
tracker.start()
try:
    # Wbudowana pętla: epoki -> batche -> loss -> backward -> optimizer.step -> walidacja.
    model.train(
        data=str(DATA), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        device=DEVICE, workers=0, seed=SEED, deterministic=True,
        optimizer="AdamW", lr0=0.001, cos_lr=True, patience=10,
        amp=torch.cuda.is_available(), cache=False,
        project=str(RUN.parent), name=RUN.name, exist_ok=True,
        save=True, plots=True, val=True,
    )
finally:
    co2_kg = tracker.stop()  # Zapis również przy błędzie lub przerwaniu treningu.
    print("Szacowane CO₂eq [kg]:", co2_kg)

## Wyniki i końcowy test
`weights/best.pt` jest wybierany na zbiorze **valid**; **test** służy wyłącznie końcowej ocenie.
`results.csv` zawiera loss i metryki kolejnych epok, `emissions.csv` zawiera CO₂eq [kg], energię [kWh] i czas [s].
Wagi `best.pt` są punktem wyjścia do późniejszego eksportu mobilnego i kwantyzacji.

In [ ]:
best = RUN / "weights/best.pt"
assert best.is_file(), "Brak best.pt — trening musi zakończyć się zapisaniem modelu."
metrics = YOLO(str(best)).val(
    data=str(DATA), split="test", imgsz=IMGSZ, batch=BATCH,
    device=DEVICE, workers=0, project=str(RUN), name="test", plots=True,
)
scores = {"mAP50": float(metrics.box.map50), "mAP50-95": float(metrics.box.map),
          "precision": float(metrics.box.mp), "recall": float(metrics.box.mr)}
(RUN / "test_metrics.json").write_text(json.dumps(scores, indent=2))
print(scores)
display(pd.read_csv(RUN / "results.csv").tail())
if (RUN / "emissions.csv").is_file():
    display(pd.read_csv(RUN / "emissions.csv")[["emissions", "energy_consumed", "duration"]])
else:
    print("Brak raportu CodeCarbon — sprawdź komunikaty trackera; brak pomiaru nie oznacza zerowej emisji.")